In [1]:
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"사용 중인 GPU: {torch.cuda.get_device_name(0)}")

PyTorch 버전: 2.5.1
CUDA 사용 가능 여부: True
사용 중인 GPU: NVIDIA RTX A4000


In [2]:
# 데이터 처리
import pandas as pd
import numpy as np

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 텍스트 탐색용
import re
from collections import Counter

# 머신러닝 (scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# 모델
from xgboost import XGBClassifier

# 희소 행렬(sparse matrix) 타입 확인용
from scipy import sparse

# 그래프에서 한글이 깨지지 않도록 폰트 설정 (Windows 기준: 맑은 고딕)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지

#plt 그래프 스타일 설정
plt.style.use("ggplot")

# 노트북 안에서 그래프가 바로 보이도록 설정
%matplotlib inline

print("라이브러리 불러오기 완료")

라이브러리 불러오기 완료


In [3]:
# ---- 파일 경로 설정 ----
# 노트북과 같은 폴더에 있는 파일명을 기준으로 경로를 설정합니다.
# 파일을 다른 폴더에 저장했거나 파일명이 다르면, 아래 값을 실제 파일 위치에 맞게 수정합니다.
# 이후 코드들은 파일명을 직접 쓰지 않고 이 변수들을 사용하므로, 경로 변경은 이 셀만 수정하면 됩니다.
TRAIN_PATH = "security_log_train.csv"             # 학습 데이터 (로그 텍스트 + 정답 level)
TEST_PATH = "security_log_test.csv"               # 테스트 데이터 (정답 level 없음)
SAMPLE_SUBMISSION_PATH = "sample_submission.csv"  # 제출 양식
OUTPUT_PATH = "submission.csv"                     # 생성할 제출 파일

# ---- 실행 옵션 ----
# RANDOM_STATE: 결과 재현성을 위한 난수 시드입니다.
# 같은 시드를 사용하면 train/valid 분리나 모델 학습 결과가 매번 동일하게 재현됩니다.
RANDOM_STATE = 42

print("설정 완료 (전체 train 데이터를 사용합니다)")

설정 완료 (전체 train 데이터를 사용합니다)


In [4]:
# 위에서 정의한 경로 변수를 사용해 세 개의 데이터를 불러옵니다.
# train: 모델이 학습할 로그 텍스트(full_log)와 정답 라벨(level)이 들어 있습니다.
# test : 예측 대상 데이터입니다. 정답 level이 없으므로 우리가 예측해서 채웁니다.
# sample_submission: 제출 파일의 형식(행 개수, 컬럼)을 알려주는 양식입니다.
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

# 각 데이터의 행/열 개수를 확인합니다. (train과 test의 행 개수는 보통 다릅니다.)
print("train shape:", train.shape)
print("test shape :", test.shape)
print("sample_submission shape:", sample_submission.shape)

train shape: (472972, 3)
test shape : (1418916, 2)
sample_submission shape: (1418916, 2)


In [5]:
import re
import os
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

def text_mining(df):
    df_ = df.copy()
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'(\\n)',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[^a-zA-Zㄱ-ㅣ가-힣0-9:=\s\(\)./,\<\>]+','',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r' ?(?P<note>[:=\(\)./,\<\>]) ?', ' \g<note> ', x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[0-9]+','<num>',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\s+',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[a-zA-Z]+\<','<eng><',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\>[a-zA-Z]+','><eng>',x))

    return df_

# 2. 저장 폴더 설정
model_dir = "./model"
os.makedirs(model_dir, exist_ok=True)

print("시작: ⭐️ 10단계 제안 조합(IP + 숫자 + 날짜시간)의 Macro F1 검증을 시작합니다.")

# 3. 데이터 복사 및 10단계 마스킹 적용
df_temp = train.copy()
df_temp = text_mining(df_temp)

# 4. Train / Valid 분리 (8:2 비율)
X_text = df_temp["full_log"]
y_labels = df_temp["level"]
X_train_t, X_valid_t, y_train_t, y_valid_t = train_test_split(
    X_text, y_labels, test_size=0.2, random_state=RANDOM_STATE, stratify=y_labels
)

# 5. TF-IDF 벡터라이저 피처 생성
tfidf_vec = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    token_pattern=r"(?u)<\w+>|\b\w\w+\b"
)

X_train_tfidf = tfidf_vec.fit_transform(X_train_t).astype('float32')
X_valid_tfidf = tfidf_vec.transform(X_valid_t).astype('float32')

labels_sorted = sorted(train["level"].unique())
num_classes = len(labels_sorted)

# 6. XGBoost 모델 정의 및 학습 (GPU 가속 사용)
model = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
    objective="multi:softprob", num_class=num_classes, eval_metric="mlogloss",
    min_child_weight= 1.0, reg_alpha= 0.01, reg_lambda= 0.5,
    random_state=RANDOM_STATE, tree_method="hist", device="cuda"
)
model.fit(X_train_tfidf, y_train_t)

# 7. 10단계 모델 및 벡터라이저 저장
model.save_model(os.path.join(model_dir, "last_mode.json"))
with open(os.path.join(model_dir, "tfidf_vec_last_mode.pkl"), "wb") as f:
    pickle.dump(tfidf_vec, f)
    
print("-> [저장 완료] ./model/ 폴더에 mode_10 모델과 벡터라이저가 저장되었습니다.")

# 8. 검증 데이터를 통한 예측 및 최종 Macro F1 계산
valid_pred = model.predict(X_valid_tfidf)
macro_f1 = f1_score(y_valid_t, valid_pred, labels=labels_sorted, average="macro", zero_division=0)

# 9. 결과 출력
print("\n========================= [10단계 검증 결과 리포트] =========================")
print(f"🎯 검증 성능 (Macro F1): {macro_f1:.4f}")
print("---------------------------------------------------------------------")
print("[검증 데이터 실제 정답 분포]")
print(y_valid_t.value_counts().sort_index())
print("\n[검증 데이터 모델 예측 분포]")
print(pd.Series(valid_pred).value_counts().sort_index())
print("=====================================================================")

시작: ⭐️ 10단계 제안 조합(IP + 숫자 + 날짜시간)의 Macro F1 검증을 시작합니다.
-> [저장 완료] ./model/ 폴더에 mode_10 모델과 벡터라이저가 저장되었습니다.


c:\Users\user\anaconda3\envs\ml_env01\Lib\site-packages\xgboost\core.py:751: UserWarning: [14:28:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



========================= [10단계 검증 결과 리포트] =========================
🎯 검증 성능 (Macro F1): 0.9480
---------------------------------------------------------------------
[검증 데이터 실제 정답 분포]
level
0    66813
1    26504
2        2
3      828
4        2
5      444
6        2
Name: count, dtype: int64

[검증 데이터 모델 예측 분포]
0    66966
1    26377
2        2
3      823
4        2
5      424
6        1
Name: count, dtype: int64


In [6]:
import re
import os
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

# GPU 사용 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 디바이스: {device}")

def text_mining(df):
    df_ = df.copy()
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'(\\n)',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[^a-zA-Zㄱ-ㅣ가-힣0-9:=\s\(\)./,\<\>]+','',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r' ?(?P<note>[:=\(\)./,\<\>]) ?', ' \g<note> ', x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[0-9]+','<num>',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\s+',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[a-zA-Z]+\<','<eng><',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\>[a-zA-Z]+','><eng>',x))

    return df_

print("시작: 5번 딥러닝 [Embedding + LSTM] 실험을 시작합니다.")

# 2. 데이터 복사 및 마스킹 적용 (기존과 100% 동일한 데이터 분할 유지)
df_temp = train.copy()
df_temp = text_mining(df_temp)

X_text = df_temp["full_log"]
y_labels = df_temp["level"]
X_train_t, X_valid_t, y_train_t, y_valid_t = train_test_split(
    X_text, y_labels, test_size=0.2, random_state=RANDOM_STATE, stratify=y_labels
)

# =========================================================================
# [중요] 3. 텍스트 토큰화 및 단어 사전(Vocabulary) 구축
# =========================================================================
# TF-IDF와 동일한 토큰화 규칙 적용 (<태그> 보존 및 일반 단어 추출)
token_pattern = re.compile(r"(?u)<\w+>|\b\w\w+\b")

def tokenize(text):
    return token_pattern.findall(text.lower())

print("-> 단어 사전 생성 중... (시간이 수십 초 소요될 수 있습니다)")
all_tokens = []
for text in X_train_t:
    all_tokens.extend(tokenize(text))

# TF-IDF 스펙과 동일하게 설정: 최대 10,000개 단어, 최소 빈도 3회
max_features = 10000
min_df = 3

word_counts = Counter(all_tokens)
vocab = {"<PAD>": 0, "<UNK>": 1} # 패딩 토큰과 모르는 단어 토큰 설정

for word, count in word_counts.most_common(max_features - 2):
    if count >= min_df:
        vocab[word] = len(vocab)

vocab_size = len(vocab)
print(f"-> 최종 단어 사전 크기: {vocab_size}")

# 나중에 4번 앙상블 및 테스트 데이터 예측을 위해 보관
model_dir = "./model"
os.makedirs(model_dir, exist_ok=True)
with open(os.path.join(model_dir, "vocab_last_mode.pkl"), "wb") as f:
    pickle.dump(vocab, f)

# =========================================================================
# 4. PyTorch 전용 커스텀 Dataset 및 DataLoader 정의
# =========================================================================
class LogDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=128):
        self.labels = labels.values if hasattr(labels, 'values') else labels
        self.sequences = []
        
        for text in texts:
            tokens = tokenize(text)
            # 단어 사전에 있으면 인덱스 반환, 없으면 <UNK> 반환
            seq = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
            
            # 고정된 길이(max_len)로 패딩 및 잘라내기(Truncation)
            if len(seq) < max_len:
                seq = seq + [vocab["<PAD>"]] * (max_len - len(seq))
            else:
                seq = seq[:max_len]
            self.sequences.append(seq)
            
        self.sequences = torch.tensor(self.sequences, dtype=torch.long)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

# 하이퍼파라미터 설정
MAX_LEN = 128       # 로그의 핵심 문맥을 보존할 수 있는 길이
BATCH_SIZE = 512    # GPU 가속을 극대화하기 위한 큰 배치 사이즈

print("-> PyTorch Dataset 변환 중...")
train_dataset = LogDataset(X_train_t, y_train_t, vocab, max_len=MAX_LEN)
valid_dataset = LogDataset(X_valid_t, y_valid_t, vocab, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

# =========================================================================
# 5. Embedding + LSTM 모델 아키텍처 정의
# =========================================================================
class EmbeddingLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_size=128, num_classes=7):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        # batch_first=True를 주어야 [Batch, Seq, Feature] 형태로 입력받습니다.
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        emb = self.embedding(x)  # [Batch, Max_Len, Embedding_Dim]
        out, (hn, cn) = self.lstm(emb)  # out: [Batch, Max_Len, Hidden_Size]
        
        # LSTM의 마지막 타임스텝 출력(문장 전체 요약 정보)을 꺼냅니다.
        last_hidden = out[:, -1, :]  # [Batch, Hidden_Size]
        logits = self.fc(last_hidden)  # [Batch, Num_Classes]
        return logits

labels_sorted = sorted(train["level"].unique())
num_classes = len(labels_sorted)

model_dl = EmbeddingLSTM(vocab_size=vocab_size, num_classes=num_classes).to(device)

# =========================================================================
# 6. 3번 실험의 최고 가중치(Custom Weight) 이식 및 손실 함수 설정
# =========================================================================
manual_class_weights = {0: 1.0, 1: 1.0, 2: 10.0, 3: 2.0, 4: 15.0, 5: 2.0, 6: 20.0}
weight_list = [manual_class_weights.get(c, 1.0) for c in labels_sorted]
class_weights_tensor = torch.tensor(weight_list, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam(model_dl.parameters(), lr=0.003)

# =========================================================================
# 7. 딥러닝 모델 학습 루프 (Epoch 3회로 압축 실험)
# =========================================================================
epochs = 10
print("\n🔥 모델 학습을 시작합니다.")

for epoch in range(epochs):
    model_dl.train()
    total_loss = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model_dl(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}] - Avg Loss: {total_loss/len(train_loader):.4f}")

# 가중치가 학습된 딥러닝 모델 가중치 파일 저장
torch.save(model_dl.state_dict(), os.path.join(model_dir, "lstm_mode_last_custom.pth"))
print("-> [저장 완료] ./model/lstm_mode_10_custom.pth 에 모델이 저장되었습니다.")

# =========================================================================
# 8. 검증 데이터 평가 및 리포트 출력
# =========================================================================
model_dl.eval()
all_preds = []

print("\n📊 검증 데이터 평가 진행 중...")
with torch.no_grad():
    for batch_x, _ in valid_loader:
        batch_x = batch_x.to(device)
        outputs = model_dl(batch_x)
        
        # 가장 높은 확률을 가진 클래스 인덱스 추출
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)

all_preds = np.array(all_preds)

# 최종 결과 리포트
print("\n========================= [5-1번 실험: Embedding + LSTM 결과] =========================")
print(classification_report(y_valid_t, all_preds, labels=labels_sorted, zero_division=0))
macro_f1_dl = f1_score(y_valid_t, all_preds, labels=labels_sorted, average="macro", zero_division=0)
print(f"🎯 LSTM 검증 성능 (Macro F1): {macro_f1_dl:.4f}")
print("==========================================================================================")

현재 사용 중인 디바이스: cuda
시작: 5번 딥러닝 [Embedding + LSTM] 실험을 시작합니다.
-> 단어 사전 생성 중... (시간이 수십 초 소요될 수 있습니다)
-> 최종 단어 사전 크기: 6333
-> PyTorch Dataset 변환 중...

🔥 모델 학습을 시작합니다.
Epoch [1/10] - Avg Loss: 0.1470
Epoch [2/10] - Avg Loss: 0.0526
Epoch [3/10] - Avg Loss: 0.0472
Epoch [4/10] - Avg Loss: 0.0454
Epoch [5/10] - Avg Loss: 0.0426
Epoch [6/10] - Avg Loss: 0.0400
Epoch [7/10] - Avg Loss: 0.0378
Epoch [8/10] - Avg Loss: 0.0403
Epoch [9/10] - Avg Loss: 0.0395
Epoch [10/10] - Avg Loss: 0.0284
-> [저장 완료] ./model/lstm_mode_10_custom.pth 에 모델이 저장되었습니다.

📊 검증 데이터 평가 진행 중...

========================= [5-1번 실험: Embedding + LSTM 결과] =========================
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     66813
           1       0.99      1.00      0.99     26504
           2       0.22      1.00      0.36         2
           3       0.99      0.99      0.99       828
           4       0.00      0.00      0.00         2
           5       0.96   

In [7]:
import re
import os
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 디바이스: {device}")

def text_mining(df):
    df_ = df.copy()
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'(\\n)',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[^a-zA-Zㄱ-ㅣ가-힣0-9:=\s\(\)./,\<\>]+','',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r' ?(?P<note>[:=\(\)./,\<\>]) ?', ' \g<note> ', x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[0-9]+','<num>',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\s+',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[a-zA-Z]+\<','<eng><',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\>[a-zA-Z]+','><eng>',x))

    return df_

print("시작: 5-2번 딥러닝 [CNN + BiLSTM] 실험을 시작합니다.")

# 2. 데이터 복사 및 마스킹 적용
df_temp = train.copy()
df_temp = text_mining(df_temp)

X_text = df_temp["full_log"]
y_labels = df_temp["level"]
X_train_t, X_valid_t, y_train_t, y_valid_t = train_test_split(
    X_text, y_labels, test_size=0.2, random_state=RANDOM_STATE, stratify=y_labels
)

# 3. 기존에 저장했던 단어 사전(Vocab) 로드
model_dir = "./model"
with open(os.path.join(model_dir, "vocab_last_mode.pkl"), "rb") as f:
    vocab = pickle.load(f)
vocab_size = len(vocab)

# 4. Dataset 정의 (동일 스펙)
token_pattern = re.compile(r"(?u)<\w+>|\b\w\w+\b")
def tokenize(text):
    return token_pattern.findall(text.lower())

class LogDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=128):
        self.labels = labels.values if hasattr(labels, 'values') else labels
        self.sequences = []
        for text in texts:
            tokens = tokenize(text)
            seq = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
            if len(seq) < max_len:
                seq = seq + [vocab["<PAD>"]] * (max_len - len(seq))
            else:
                seq = seq[:max_len]
            self.sequences.append(seq)
        self.sequences = torch.tensor(self.sequences, dtype=torch.long)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.sequences[idx], self.labels[idx]

MAX_LEN = 128
BATCH_SIZE = 512

train_dataset = LogDataset(X_train_t, y_train_t, vocab, max_len=MAX_LEN)
valid_dataset = LogDataset(X_valid_t, y_valid_t, vocab, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

# =========================================================================
# 5. CNN + BiLSTM 아키텍처 정의 (패딩 킬러 구조 🎯)
# =========================================================================
class CNNBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, num_filters=128, hidden_size=128, num_classes=7):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 지역적 키워드 조합(N-gram)을 추출하는 1D CNN 문맥 필터
        self.conv = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        
        # 순서 관계를 파악하는 양방향 LSTM
        self.lstm = nn.LSTM(num_filters, hidden_size, num_layers=1, batch_first=True, bidirectional=True)
        
        # 최종 분류 레이어 (BiLSTM이므로 hidden_size * 2 차원)
        self.fc = nn.Linear(hidden_size * 2, num_classes)
        
    def forward(self, x):
        # 1. Embedding 차원 변환: [Batch, Length, Dim] -> [Batch, Dim, Length] (Conv1d 입력 요구 조건)
        emb = self.embedding(x).transpose(1, 2)
        
        # 2. CNN 필터 통과
        conv_out = self.relu(self.conv(emb)).transpose(1, 2) # 다시 [Batch, Length, Filters] 로 복복귀
        
        # 3. BiLSTM 통과
        lstm_out, _ = self.lstm(conv_out) # [Batch, Length, Hidden_Size * 2]
        
        # 4. 🔥 [핵심] Global Max Pooling 적용하여 패딩 문자열 무력화!
        # 문장 전체(Length 축)를 통틀어 가장 강하게 활성화된 위협 신호 피처만 탑 슬라이싱
        lstm_out = lstm_out.transpose(1, 2) # [Batch, Hidden_Size*2, Length]
        pooled = torch.max(lstm_out, dim=2)[0] # [Batch, Hidden_Size*2]
        
        # 5. 최종 예측 확률 도출
        logits = self.fc(pooled)
        return logits

labels_sorted = sorted(train["level"].unique())
num_classes = len(labels_sorted)

model_cnn_lstm = CNNBiLSTM(vocab_size=vocab_size, num_classes=num_classes).to(device)

# 6. 동일한 Custom 가중치 이식
manual_class_weights = {0: 1.0, 1: 1.0, 2: 10.0, 3: 2.0, 4: 15.0, 5: 2.0, 6: 20.0}
weight_list = [manual_class_weights.get(c, 1.0) for c in labels_sorted]
class_weights_tensor = torch.tensor(weight_list, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam(model_cnn_lstm.parameters(), lr=0.003)

# 7. 모델 학습 루프 (수렴 속도가 빨라 똑같이 3 epoch만 진행해도 점수가 확 달라집니다)
epochs = 3
print("\n🔥 [CNN + BiLSTM] 모델 학습을 시작합니다.")

for epoch in range(epochs):
    model_cnn_lstm.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model_cnn_lstm(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/{epochs}] - Avg Loss: {total_loss/len(train_loader):.4f}")

# 모델 저장
torch.save(model_cnn_lstm.state_dict(), os.path.join(model_dir, "cnn_lstm_mode_last_custom.pth"))
print("-> [저장 완료] ./model/cnn_lstm_mode_10_custom.pth 에 모델이 저장되었습니다.")

# 8. 검증 데이터 평가
model_cnn_lstm.eval()
all_preds = []

print("\n📊 검증 데이터 평가 진행 중...")
with torch.no_grad():
    for batch_x, _ in valid_loader:
        batch_x = batch_x.to(device)
        outputs = model_cnn_lstm(batch_x)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)

all_preds = np.array(all_preds)

print("\n========================= [5-2번 실험: CNN + BiLSTM 결과] =========================")
print(classification_report(y_valid_t, all_preds, labels=labels_sorted, zero_division=0))
macro_f1_cnn_lstm = f1_score(y_valid_t, all_preds, labels=labels_sorted, average="macro", zero_division=0)
print(f"🎯 CNN + BiLSTM 검증 성능 (Macro F1): {macro_f1_cnn_lstm:.4f}")
print("==========================================================================================")

현재 사용 중인 디바이스: cuda
시작: 5-2번 딥러닝 [CNN + BiLSTM] 실험을 시작합니다.

🔥 [CNN + BiLSTM] 모델 학습을 시작합니다.
Epoch [1/3] - Avg Loss: 0.0263
Epoch [2/3] - Avg Loss: 0.0059
Epoch [3/3] - Avg Loss: 0.0052
-> [저장 완료] ./model/cnn_lstm_mode_10_custom.pth 에 모델이 저장되었습니다.

📊 검증 데이터 평가 진행 중...

========================= [5-2번 실험: CNN + BiLSTM 결과] =========================
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     66813
           1       1.00      0.99      1.00     26504
           2       1.00      1.00      1.00         2
           3       1.00      0.99      1.00       828
           4       1.00      0.50      0.67         2
           5       1.00      0.95      0.97       444
           6       1.00      0.50      0.67         2

    accuracy                           1.00     94595
   macro avg       1.00      0.85      0.90     94595
weighted avg       1.00      1.00      1.00     94595

🎯 CNN + BiLSTM 검증 성능 (Macro F1): 0.9001


In [8]:
import re
import os
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 디바이스: {device}")

def text_mining(df):
    df_ = df.copy()
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'(\\n)',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[^a-zA-Zㄱ-ㅣ가-힣0-9:=\s\(\)./,\<\>]+','',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r' ?(?P<note>[:=\(\)./,\<\>]) ?', ' \g<note> ', x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[0-9]+','<num>',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\s+',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[a-zA-Z]+\<','<eng><',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\>[a-zA-Z]+','><eng>',x))

    return df_

print("시작: 5-3번 딥러닝 [Pure 1D-CNN] 실험을 시작합니다.")

# 2. 데이터 복사 및 마스킹 적용 (기존과 100% 동일한 데이터 분할 유지)
df_temp = train.copy()
df_temp = text_mining(df_temp)

X_text = df_temp["full_log"]
y_labels = df_temp["level"]
X_train_t, X_valid_t, y_train_t, y_valid_t = train_test_split(
    X_text, y_labels, test_size=0.2, random_state=RANDOM_STATE, stratify=y_labels
)

# 3. 기존에 저장했던 단어 사전(Vocab) 로드
model_dir = "./model"
with open(os.path.join(model_dir, "vocab_last_mode.pkl"), "rb") as f:
    vocab = pickle.load(f)
vocab_size = len(vocab)

# 4. Dataset 정의 (동일 스펙 고정)
token_pattern = re.compile(r"(?u)<\w+>|\b\w\w+\b")
def tokenize(text):
    return token_pattern.findall(text.lower())

class LogDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=128):
        self.labels = labels.values if hasattr(labels, 'values') else labels
        self.sequences = []
        for text in texts:
            tokens = tokenize(text)
            seq = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
            if len(seq) < max_len:
                seq = seq + [vocab["<PAD>"]] * (max_len - len(seq))
            else:
                seq = seq[:max_len]
            self.sequences.append(seq)
        self.sequences = torch.tensor(self.sequences, dtype=torch.long)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.sequences[idx], self.labels[idx]

MAX_LEN = 128
BATCH_SIZE = 512

train_dataset = LogDataset(X_train_t, y_train_t, vocab, max_len=MAX_LEN)
valid_dataset = LogDataset(X_valid_t, y_valid_t, vocab, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

# =========================================================================
# 5. Pure 1D-CNN 아키텍처 정의 (핵심 키워드 추출 스나이퍼 🎯)
# =========================================================================
class Pure1DCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, num_filters=256, num_classes=7):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # kernel_size=5로 확장하여 조금 더 긴 위험 문맥 단어 조합을 포착합니다.
        self.conv = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        
        # 바로 분류 레이어로 연결 (중간에 LSTM 연산이 없어 엄청나게 가볍고 빠름)
        self.fc = nn.Linear(num_filters, num_classes)
        
    def forward(self, x):
        # [Batch, Length, Dim] -> [Batch, Dim, Length]
        emb = self.embedding(x).transpose(1, 2)
        
        # 특징 맵 추출
        conv_out = self.relu(self.conv(emb))
        
        # Global Max Pooling: 문장에서 필터별로 가장 강한 신호 딱 1개씩만 복제 [Batch, Filters]
        pooled = torch.max(conv_out, dim=2)[0]
        
        # 과적합 방지 후 최종 출력
        pooled = self.dropout(pooled)
        logits = self.fc(pooled)
        return logits

labels_sorted = sorted(train["level"].unique())
num_classes = len(labels_sorted)

model_pure_cnn = Pure1DCNN(vocab_size=vocab_size, num_classes=num_classes).to(device)

# 6. 동일한 Custom 가중치 이식
manual_class_weights = {0: 1.0, 1: 1.0, 2: 10.0, 3: 2.0, 4: 15.0, 5: 2.0, 6: 20.0}
weight_list = [manual_class_weights.get(c, 1.0) for c in labels_sorted]
class_weights_tensor = torch.tensor(weight_list, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam(model_pure_cnn.parameters(), lr=0.003)

# 7. 모델 학습 루프 (매우 빠른 속도로 학습이 끝납니다)
epochs = 3
print("\n🔥 [Pure 1D-CNN] 모델 학습을 시작합니다.")

for epoch in range(epochs):
    model_pure_cnn.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model_pure_cnn(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/{epochs}] - Avg Loss: {total_loss/len(train_loader):.4f}")

# 모델 저장
torch.save(model_pure_cnn.state_dict(), os.path.join(model_dir, "pure_cnn_mode_last_custom.pth"))
print("-> [저장 완료] ./model/pure_cnn_mode_last_custom.pth 에 모델이 저장되었습니다.")

# 8. 검증 데이터 평가
model_pure_cnn.eval()
all_preds = []

print("\n📊 검증 데이터 평가 진행 중...")
with torch.no_grad():
    for batch_x, _ in valid_loader:
        batch_x = batch_x.to(device)
        outputs = model_pure_cnn(batch_x)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)

all_preds = np.array(all_preds)

print("\n========================= [5-3번 실험: Pure 1D-CNN 결과] =========================")
print(classification_report(y_valid_t, all_preds, labels=labels_sorted, zero_division=0))
macro_f1_pure_cnn = f1_score(y_valid_t, all_preds, labels=labels_sorted, average="macro", zero_division=0)
print(f"🎯 Pure 1D-CNN 검증 성능 (Macro F1): {macro_f1_pure_cnn:.4f}")
print("==========================================================================================")

현재 사용 중인 디바이스: cuda
시작: 5-3번 딥러닝 [Pure 1D-CNN] 실험을 시작합니다.

🔥 [Pure 1D-CNN] 모델 학습을 시작합니다.
Epoch [1/3] - Avg Loss: 0.0167
Epoch [2/3] - Avg Loss: 0.0072
Epoch [3/3] - Avg Loss: 0.0097
-> [저장 완료] ./model/pure_cnn_mode_last_custom.pth 에 모델이 저장되었습니다.

📊 검증 데이터 평가 진행 중...

========================= [5-3번 실험: Pure 1D-CNN 결과] =========================
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     66813
           1       1.00      0.99      1.00     26504
           2       1.00      1.00      1.00         2
           3       1.00      0.99      1.00       828
           4       1.00      1.00      1.00         2
           5       1.00      0.95      0.97       444
           6       1.00      0.50      0.67         2

    accuracy                           1.00     94595
   macro avg       1.00      0.92      0.95     94595
weighted avg       1.00      1.00      1.00     94595

🎯 Pure 1D-CNN 검증 성능 (Macro F1): 0.9474


In [9]:
import re
import os
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dir = "./model"

def text_mining(df):
    df_ = df.copy()
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'(\\n)',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[^a-zA-Zㄱ-ㅣ가-힣0-9:=\s\(\)./,\<\>]+','',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r' ?(?P<note>[:=\(\)./,\<\>]) ?', ' \g<note> ', x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[0-9]+','<num>',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\s+',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[a-zA-Z]+\<','<eng><',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\>[a-zA-Z]+','><eng>',x))

    return df_

print("시작: 4번 [머신러닝 + 딥러닝 3대 모델 앙상블]을 진행합니다.")

# 2. 데이터 복사 및 검증 데이터셋 준비 (기존 구조 보존)
df_temp = train.copy()
df_temp = text_mining(df_temp)

X_text = df_temp["full_log"]
y_labels = df_temp["level"]
_, X_valid_t, _, y_valid_t = train_test_split(
    X_text, y_labels, test_size=0.2, random_state=RANDOM_STATE, stratify=y_labels
)

labels_sorted = sorted(train["level"].unique())
num_classes = len(labels_sorted)

# =========================================================================
# [모델 1 준비] Custom XGBoost 로드 및 확률 예측
# =========================================================================
print("\n[1/3] Custom XGBoost 모델 로드 및 확률 계산 중...")
with open(os.path.join(model_dir, "tfidf_vec_last_mode.pkl"), "rb") as f:
    tfidf_vec = pickle.load(f)
X_valid_tfidf = tfidf_vec.transform(X_valid_t).astype('float32')

model_xgb = XGBClassifier()
model_xgb.load_model(os.path.join(model_dir, "last_mode.json"))
proba_xgb = model_xgb.predict_proba(X_valid_tfidf) # shape: [N, 7]


# =========================================================================
# [딥러닝 공통 준비] 단어 사전 및 데이터로더 세팅
# =========================================================================
with open(os.path.join(model_dir, "vocab_last_mode.pkl"), "rb") as f:
    vocab = pickle.load(f)
vocab_size = len(vocab)

token_pattern = re.compile(r"(?u)<\w+>|\b\w\w+\b")
def tokenize(text): return token_pattern.findall(text.lower())

class LogDataset(Dataset):
    def __init__(self, texts, vocab, max_len=128):
        self.sequences = []
        for text in texts:
            tokens = tokenize(text)
            seq = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
            if len(seq) < max_len: seq = seq + [vocab["<PAD>"]] * (max_len - len(seq))
            else: seq = seq[:max_len]
            self.sequences.append(seq)
        self.sequences = torch.tensor(self.sequences, dtype=torch.long)
    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx): return self.sequences[idx]

valid_dataset = LogDataset(X_valid_t, vocab, max_len=128)
valid_loader = DataLoader(valid_dataset, batch_size=512, shuffle=False)


# =========================================================================
# [모델 2 준비] CNN + BiLSTM 로드 및 확률 예측
# =========================================================================
print("[2/3] CNN + BiLSTM 모델 로드 및 확률 계산 중...")
class CNNBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, num_filters=128, hidden_size=128, num_classes=7):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.conv = nn.Conv1d(embedding_dim, num_filters, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(num_filters, hidden_size, num_layers=1, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, num_classes)
    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)
        conv_out = self.relu(self.conv(emb)).transpose(1, 2)
        lstm_out, _ = self.lstm(conv_out)
        pooled = torch.max(lstm_out.transpose(1, 2), dim=2)[0]
        return self.fc(pooled)

model_cnn_lstm = CNNBiLSTM(vocab_size=vocab_size, num_classes=num_classes).to(device)
model_cnn_lstm.load_state_dict(torch.load(os.path.join(model_dir, "cnn_lstm_mode_last_custom.pth"), map_location=device))
model_cnn_lstm.eval()

proba_cnn_lstm = []
with torch.no_grad():
    for batch_x in valid_loader:
        outputs = model_cnn_lstm(batch_x.to(device))
        # 오차 계산을 위해 Softmax를 취해 "확률(0~1)"로 변환합니다.
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        proba_cnn_lstm.extend(probs)
proba_cnn_lstm = np.array(proba_cnn_lstm) # shape: [N, 7]


# =========================================================================
# [모델 3 준비] Pure 1D-CNN 로드 및 확률 예측
# =========================================================================
print("[3/3] Pure 1D-CNN 모델 로드 및 확률 계산 중...")
class Pure1DCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, num_filters=256, num_classes=7):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.conv = nn.Conv1d(embedding_dim, num_filters, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(num_filters, num_classes)
    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)
        conv_out = self.relu(self.conv(emb))
        pooled = torch.max(conv_out, dim=2)[0]
        return self.fc(self.dropout(pooled))

model_pure_cnn = Pure1DCNN(vocab_size=vocab_size, num_classes=num_classes).to(device)
model_pure_cnn.load_state_dict(torch.load(os.path.join(model_dir, "pure_cnn_mode_last_custom.pth"), map_location=device))
model_pure_cnn.eval()

proba_pure_cnn = []
with torch.no_grad():
    for batch_x in valid_loader:
        outputs = model_pure_cnn(batch_x.to(device))
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        proba_pure_cnn.extend(probs)
proba_pure_cnn = np.array(proba_pure_cnn) # shape: [N, 7]


# =========================================================================
# 5. 대망의 Soft Voting (확률 평균 결합) 연산
# =========================================================================
print("\n🔥 3대 모델의 지혜를 모아 소프트 보팅 앙상블을 수행합니다...")

# 기본 균등 결합 (1:1:1비중)
final_proba = (proba_xgb + proba_cnn_lstm + proba_pure_cnn) / 3.0

# 최종 결론 선택 (가장 높은 평균 확률을 가진 인덱스 추출)
final_preds = np.argmax(final_proba, axis=1)

# =========================================================================
# 6. 최종 앙상블 결과 리포트 출력
# =========================================================================
print("\n========================= [4번 결과: 3대 모델 집단지성 앙상블 리포트] =========================")
print(classification_report(y_valid_t, final_preds, labels=labels_sorted, zero_division=0))
macro_f1_ensemble = f1_score(y_valid_t, final_preds, labels=labels_sorted, average="macro", zero_division=0)
print(f"🏆 앙상블 모델 최종 검증 성능 (Macro F1): {macro_f1_ensemble:.4f}")
print("==========================================================================================")

시작: 4번 [머신러닝 + 딥러닝 3대 모델 앙상블]을 진행합니다.

[1/3] Custom XGBoost 모델 로드 및 확률 계산 중...
[2/3] CNN + BiLSTM 모델 로드 및 확률 계산 중...


C:\Users\user\AppData\Local\Temp\ipykernel_7772\2800710082.py:103: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_cnn_lstm.load_state_dict(torch.load(os.path.join(model

[3/3] Pure 1D-CNN 모델 로드 및 확률 계산 중...


C:\Users\user\AppData\Local\Temp\ipykernel_7772\2800710082.py:135: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_pure_cnn.load_state_dict(torch.load(os.path.join(model


🔥 3대 모델의 지혜를 모아 소프트 보팅 앙상블을 수행합니다...

========================= [4번 결과: 3대 모델 집단지성 앙상블 리포트] =========================
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     66813
           1       1.00      0.99      1.00     26504
           2       1.00      1.00      1.00         2
           3       1.00      0.99      1.00       828
           4       1.00      1.00      1.00         2
           5       1.00      0.95      0.98       444
           6       1.00      0.50      0.67         2

    accuracy                           1.00     94595
   macro avg       1.00      0.92      0.95     94595
weighted avg       1.00      1.00      1.00     94595

🏆 앙상블 모델 최종 검증 성능 (Macro F1): 0.9480


In [10]:
import re
import os
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dir = "./model"

def text_mining(df):
    df_ = df.copy()
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'(\\n)',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[^a-zA-Zㄱ-ㅣ가-힣0-9:=\s\(\)./,\<\>]+','',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r' ?(?P<note>[:=\(\)./,\<\>]) ?', ' \g<note> ', x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[0-9]+','<num>',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\s+',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[a-zA-Z]+\<','<eng><',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\>[a-zA-Z]+','><eng>',x))

    return df_

print("시작: 마스터 파이프라인의 최종장 [2번 단계] 가중치 최적화 및 Test 추론을 시작합니다.")

# =========================================================================
# 1. 딥러닝 모델 아키텍처 재선언 (로드용)
# =========================================================================
class CNNBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, num_filters=128, hidden_size=128, num_classes=7):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.conv = nn.Conv1d(embedding_dim, num_filters, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(num_filters, hidden_size, num_layers=1, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, num_classes)
    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)
        conv_out = self.relu(self.conv(emb)).transpose(1, 2)
        lstm_out, _ = self.lstm(conv_out)
        pooled = torch.max(lstm_out.transpose(1, 2), dim=2)[0]
        return self.fc(pooled)

class Pure1DCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, num_filters=256, num_classes=7):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.conv = nn.Conv1d(embedding_dim, num_filters, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(num_filters, num_classes)
    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)
        conv_out = self.relu(self.conv(emb))
        pooled = torch.max(conv_out, dim=2)[0]
        return self.fc(self.dropout(pooled))

class LogDataset(Dataset):
    def __init__(self, texts, vocab, max_len=128):
        self.sequences = []
        token_pattern = re.compile(r"(?u)<\w+>|\b\w\w+\b")
        for text in texts:
            tokens = token_pattern.findall(text.lower())
            seq = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
            if len(seq) < max_len: seq = seq + [vocab["<PAD>"]] * (max_len - len(seq))
            else: seq = seq[:max_len]
            self.sequences.append(seq)
        self.sequences = torch.tensor(self.sequences, dtype=torch.long)
    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx): return self.sequences[idx]

# =========================================================================
# 2. 검증 데이터셋 가중치 미세 조정 (6번 클래스 완전 구출 작전)
# =========================================================================
# 6번을 유일하게 완벽하게 잡았던 CNN_BiLSTM의 비중을 살짝 높여 시너지를 극대화합니다.
w_xgb = 0.30
w_cnn_lstm = 0.35  # 6번 클래스 마스터의 비중 상향
w_pure_cnn = 0.35

final_proba_tuned = (proba_xgb * w_xgb) + (proba_cnn_lstm * w_cnn_lstm) + (proba_pure_cnn * w_pure_cnn)
final_preds_tuned = np.argmax(final_proba_tuned, axis=1)

print("\n========================= [2번 결과: 가중치 최적화 앙상블 리포트] =========================")
print(classification_report(y_valid_t, final_preds_tuned, labels=labels_sorted, zero_division=0))
macro_f1_tuned = f1_score(y_valid_t, final_preds_tuned, labels=labels_sorted, average="macro", zero_division=0)
print(f"🔥 가중치 튜닝 후 앙상블 성능 (Macro F1): {macro_f1_tuned:.4f}")
print("==========================================================================================")

# =========================================================================
# 3. [최종 단계] 진짜 실제 테스트 데이터(test.csv) 로드 및 마스킹
# =========================================================================
print("\n[이동] 실제 Test 데이터셋 예측 및 제출 파일 생성 단계를 시작합니다.")
df_test = test.copy()  # 최초 로드된 test 데이터 사용
df_test = text_mining(df_test)
X_test_text = df_test["full_log"]

# 3-1. XGBoost 전용 TF-IDF 변환 및 예측 확률 도출
X_test_tfidf = tfidf_vec.transform(X_test_text).astype('float32')
test_proba_xgb = model_xgb.predict_proba(X_test_tfidf)

# 3-2. 딥러닝 전용 데이터로더 생성
test_dataset = LogDataset(X_test_text, vocab, max_len=128)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

# 3-3. CNN + BiLSTM 테스트 확률 도출
test_proba_cnn_lstm = []
with torch.no_grad():
    for batch_x in test_loader:
        outputs = model_cnn_lstm(batch_x.to(device))
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        test_proba_cnn_lstm.extend(probs)
test_proba_cnn_lstm = np.array(test_proba_cnn_lstm)

# 3-4. Pure 1D-CNN 테스트 확률 도출
test_proba_pure_cnn = []
with torch.no_grad():
    for batch_x in test_loader:
        outputs = model_pure_cnn(batch_x.to(device))
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        test_proba_pure_cnn.extend(probs)
test_proba_pure_cnn = np.array(test_proba_pure_cnn)

# =========================================================================
# 4. 검증 단계에서 확정된 황금 가중치로 최종 결합
# =========================================================================
print("-> 3대 모델의 테스트 확률을 황금 비율로 결합 중...")
test_final_proba = (test_proba_xgb * w_xgb) + (test_proba_cnn_lstm * w_cnn_lstm) + (test_proba_pure_cnn * w_pure_cnn)
test_final_preds = np.argmax(test_final_proba, axis=1)

# =========================================================================
# 5. 제출 포맷 파일 작성 및 내보내기
# =========================================================================
submission_df = pd.DataFrame({
    "id": df_test["id"],
    "level": test_final_preds
})

submission_path = "./submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"\n🎉 [최종 완료] 모든 여정이 끝났습니다! 최종 제출 파일이 '{submission_path}' 에 저장되었습니다.")
print(submission_df["level"].value_counts().sort_index())

시작: 마스터 파이프라인의 최종장 [2번 단계] 가중치 최적화 및 Test 추론을 시작합니다.

========================= [2번 결과: 가중치 최적화 앙상블 리포트] =========================
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     66813
           1       1.00      0.99      1.00     26504
           2       1.00      1.00      1.00         2
           3       1.00      0.99      1.00       828
           4       1.00      1.00      1.00         2
           5       1.00      0.95      0.98       444
           6       1.00      0.50      0.67         2

    accuracy                           1.00     94595
   macro avg       1.00      0.92      0.95     94595
weighted avg       1.00      1.00      1.00     94595

🔥 가중치 튜닝 후 앙상블 성능 (Macro F1): 0.9480

[이동] 실제 Test 데이터셋 예측 및 제출 파일 생성 단계를 시작합니다.
-> 3대 모델의 테스트 확률을 황금 비율로 결합 중...

🎉 [최종 완료] 모든 여정이 끝났습니다! 최종 제출 파일이 './submission.csv' 에 저장되었습니다.
level
0    1004083
1     395374
2         34
3      12994
4         34
5       6372
6        

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("시작: [level 7 Threshold 후처리 실험]을 시작합니다.")

# 1. 이전 단계에서 계산된 3대 모델 통합 테스트 확률(test_final_proba)을 가져옵니다.
# test_final_proba 구조: [테스트 데이터 개수, 7개 클래스(0~6)]
base_preds = np.argmax(test_final_proba, axis=1) # 임계값 적용 전 기본 예측값 (0~6)
max_probas = np.max(test_final_proba, axis=1)    # 모델이 내린 가장 높은 확신의 확률 값 (0.0 ~ 1.0)

# 2. 비교할 THRESHOLD 리스트 정의 (0.50부터 0.90까지 0.05 단위 탐색)
thresholds = [0.50, 0.55, 0.60, 0.65, 0.70]

experiment_results = []

print("\n🔍 임계값(Threshold)별 클래스 분포 및 level 7 생성 개수 확인:")
print("-" * 75)
print(f"{'Threshold':<12} | {'Level 7 Count':<15} | {'전체 데이터 대비 비율':<18}")
print("-" * 75)

# 3. 루프를 돌며 후처리 적용 및 파일 저장
for th in thresholds:
    # 🔥 [핵심 원리] 가장 높은 확신 확률마저 th보다 낮다면 -> 미지의 위협 '7'로 강제 변경!
    final_preds_th = np.where(max_probas < th, 7, base_preds)
    
    # 클래스별 분포 카운트
    unique, counts = np.unique(final_preds_th, return_counts=True)
    counts_dict = dict(zip(unique, counts))
    
    lvl7_count = counts_dict.get(7, 0)
    lvl7_ratio = (lvl7_count / len(final_preds_th)) * 100
    
    print(f"TH = {th:.2f}       | {lvl7_count:<15,} 건 | {lvl7_ratio:.3f} %")
    
    # 결과 요약 저장
    experiment_results.append({
        "Threshold": th,
        "Level_7_Count": lvl7_count,
        "Full_Distribution": counts_dict
    })
    
    # 4. 각 임계값별로 제출할 수 있도록 개별 csv 파일 자동 생성
    th_sub_df = pd.DataFrame({
        "id": df_test["id"],
        "level": final_preds_th
    })
    
    file_name = f"./submission/submission_th_{th:.2f}.csv"
    th_sub_df.to_csv(file_name, index=False)

print("-" * 75)
print("🎉 [실험 완료] 각 임계값별 제출 파일이 생성되었습니다! (예: ./submission_th_0.70.csv)")
print("이제 생성된 파일들을 대회 플랫폼에 하나씩 제출해보며 Public Score의 변화를 기록하세요.")

시작: [level 7 Threshold 후처리 실험]을 시작합니다.

🔍 임계값(Threshold)별 클래스 분포 및 level 7 생성 개수 확인:
---------------------------------------------------------------------------
Threshold    | Level 7 Count   | 전체 데이터 대비 비율      
---------------------------------------------------------------------------
TH = 0.50       | 79              건 | 0.006 %
TH = 0.55       | 426             건 | 0.030 %
TH = 0.60       | 735             건 | 0.052 %
TH = 0.65       | 1,024           건 | 0.072 %
TH = 0.70       | 1,328           건 | 0.094 %
---------------------------------------------------------------------------
🎉 [실험 완료] 각 임계값별 제출 파일이 생성되었습니다! (예: ./submission_th_0.70.csv)
이제 생성된 파일들을 대회 플랫폼에 하나씩 제출해보며 Public Score의 변화를 기록하세요.
